# AeroDelay Regression Benchmark

This notebook implements the benchmark plan from `model_planning.md` for DS108. The goal is not to build a production model; it is to verify whether the Gold-layer preprocessing and feature engineering contain useful signal for predicting departure delay minutes.

Main rules:
- Train only on departure rows.
- Target is `Departure_Delay_Reg_Target`.
- Split by time, not random split.
- Exclude leakage, identity, and raw label columns from predictors.


In [ ]:
from pathlib import Path
import os
import warnings

# Keep sklearn/joblib single-threaded for Windows sandbox stability.
os.environ.setdefault("LOKY_MAX_CPU_COUNT", "1")
os.environ.setdefault("OMP_NUM_THREADS", "1")
os.environ.setdefault("MKL_NUM_THREADS", "1")
os.environ.setdefault("OPENBLAS_NUM_THREADS", "1")

import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyRegressor
from sklearn.ensemble import ExtraTreesRegressor, HistGradientBoostingRegressor
from sklearn.impute import SimpleImputer
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error, median_absolute_error, r2_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 160)
pd.set_option("display.max_rows", 120)

CWD = Path.cwd().resolve()
PROJECT_ROOT = CWD if (CWD / "Data crawl").exists() else CWD.parent
if not (PROJECT_ROOT / "Data crawl").exists():
    raise FileNotFoundError("Cannot find project root containing 'Data crawl'. Run from repo root or Source code.")

GOLD = PROJECT_ROOT / "Data crawl" / "Gold_layer"
DEPARTURE_DIR = GOLD / "Departure"
REPORT_DIR = GOLD / "Audit" / "model_training"
REPORT_DIR.mkdir(parents=True, exist_ok=True)

TARGET = "Departure_Delay_Reg_Target"
TEST_START = pd.Timestamp("2026-03-01")
PASSENGER_ONLY = True
RANDOM_STATE = 108

print("PROJECT_ROOT:", PROJECT_ROOT)
print("DEPARTURE_DIR:", DEPARTURE_DIR)
print("REPORT_DIR:", REPORT_DIR)


## 1. Load Gold Departure Data

Use `Data crawl/Gold_layer/Departure/*_flights_departure_gold_layer.csv` instead of the master file so the origin airport can be recovered from the file name. Arrival rows are intentionally excluded for the departure-delay regression target.


In [ ]:
def load_departure_gold(departure_dir: Path) -> pd.DataFrame:
    frames = []
    for path in sorted(departure_dir.glob("*_flights_departure_gold_layer.csv")):
        airport = path.name.split("_")[0].upper()
        df = pd.read_csv(path, low_memory=False)
        df["Airport"] = airport
        frames.append(df)
    if not frames:
        raise FileNotFoundError(f"No departure Gold files found in {departure_dir}")
    return pd.concat(frames, ignore_index=True, sort=False)

df_raw = load_departure_gold(DEPARTURE_DIR)
df_raw["Scheduled_Time"] = pd.to_datetime(df_raw["Scheduled_Time"], errors="coerce")

# Safe calendar features derived from scheduled time only.
df_raw["Scheduled_Hour"] = df_raw["Scheduled_Time"].dt.hour
df_raw["Scheduled_DayOfWeek"] = df_raw["Scheduled_Time"].dt.dayofweek
df_raw["Scheduled_Month"] = df_raw["Scheduled_Time"].dt.month
df_raw["Is_Weekend"] = df_raw["Scheduled_DayOfWeek"].isin([5, 6]).astype("Int64")

mask = (
    df_raw["Record_Type"].eq("Departure")
    & df_raw[TARGET].notna()
    & df_raw["Scheduled_Time"].notna()
)
if "Exclude_From_Propagation_Training" in df_raw.columns:
    mask &= ~df_raw["Exclude_From_Propagation_Training"].fillna(False).astype(bool)
if PASSENGER_ONLY and "Category" in df_raw.columns:
    mask &= df_raw["Category"].astype("string").str.lower().eq("passenger")

df = df_raw.loc[mask].copy()
df = df.sort_values("Scheduled_Time").reset_index(drop=True)

profile = pd.DataFrame([{
    "raw_rows": len(df_raw),
    "training_rows": len(df),
    "passenger_only": PASSENGER_ONLY,
    "scheduled_min": df["Scheduled_Time"].min(),
    "scheduled_max": df["Scheduled_Time"].max(),
    "target_mean": df[TARGET].mean(),
    "target_median": df[TARGET].median(),
    "target_p95": df[TARGET].quantile(0.95),
    "target_max": df[TARGET].max(),
}])
profile.to_csv(REPORT_DIR / "01_training_dataset_profile.csv", index=False)

display(profile)
display(df["Scheduled_Time"].dt.to_period("M").value_counts().sort_index().rename("rows_by_month").reset_index())
display(df[["Airport", TARGET]].groupby("Airport").agg(rows=(TARGET, "size"), median_delay=(TARGET, "median"), p95_delay=(TARGET, lambda s: s.quantile(0.95))).reset_index())


## 2. Define Leakage-Safe Feature Sets

Ablation groups follow `model_planning.md`:
- `F0_base`: schedule, airport, categorical, and load basics.
- `F1_turnaround`: adds model-safe turnaround features.
- `F2_lag`: adds propagation lag features.
- `F3_weather`: adds weather features.


In [ ]:
LEAKAGE_OR_LABEL_COLUMNS = {
    "Departure_Delay", TARGET, "Actual_Time", "Flight_No", "Airline", "Scheduled_Tail",
    "Matched_Actual_Tail", "Swap_Match_Gap_Minutes", "Runway_Swap_Event",
    "Crawl_Date", "Status", "Checkin_Time", "Checkin_Counter", "Gate",
}

F0_BASE = [
    "Airport", "IATA", "Airline_Type", "Aircraft_Type", "Category", "Time_of_Day",
    "Scheduled_Hour", "Scheduled_DayOfWeek", "Scheduled_Month", "Is_Weekend",
    "Peak_Hour_Indicator", "Is_Special_Days", "Is_Wide_Body", "Is_First_Flight",
    "Tail_Sequence_Day", "Standard_Turnaround",
    "Airport_Load_Factor", "Number_of_Flights_in_Last_Hour", "Is_Airport_Congested",
    "Is_Parallel_Usage", "Ground_Handling_Pressure", "Taxi_Out_Congestion",
    "A_CDM_TOBT_Deficit", "Destination_Congestion_Risk", "Previous_Station_Disruption",
]

F1_TURNAROUND = [
    "Turnaround_Buffer_Model", "Tail_Stagnation_Duration_Model",
    "Turnaround_Deficit_Min", "Is_Long_Ground_Turnaround",
]

F2_LAG = [
    "Prev_Departure_Delay_Tail_1", "Prev_Departure_Delay_Tail_2",
    "Rolling_Departure_Delay_Tail_3",
    "Prev_Turnaround_Buffer_Tail_1", "Prev_Turnaround_Buffer_Tail_2",
    "Rolling_Turnaround_Buffer_Tail_3",
    "Prev_Departure_Delay_Airport_1", "Rolling_Departure_Delay_Airport_3",
]

WEATHER_FEATURES = [
    "temperature", "precipitation", "cloudcover", "pressure", "humidity", "visibility",
    "dew_point_2m", "weather_code", "cape", "lifted_index", "cloud_cover_low",
    "Visibility_SM", "Visibility_Deficit_5KM_M", "Visibility_Deficit_3SM_M",
    "Visibility_Severity_Score", "Dew_Point_Spread_C", "Wind_Sector",
    "Wind_Runway_Relative_Angle_Deg", "Wind_Kt", "Crosswind_Kt", "Headwind_Kt",
    "Tailwind_Default_Runway_Kt", "Temp_Change_1H_C", "Temp_Change_3H_C",
    "Pressure_Change_3H_Hpa", "Precip_Cumsum_1H_Mm", "Precip_Cumsum_3H_Mm",
    "Precip_Cumsum_6H_Mm", "Wind_Gust_Estimate_Kt", "Crosswind_Max_3H_Kt",
    "Gust_Variation_Kt", "Is_Rain", "Is_Heavy_Rain", "Is_Low_Visibility", "Is_Fog",
    "Is_Dewpoint_Spread_Le_1_5C", "Is_Radiation_Fog_Risk", "Is_WMO_Rain_Code",
    "Is_Below_3SM_Visibility", "Is_Below_1SM_Visibility", "Is_Crosswind_10kt",
    "Is_Tailwind_Default_Runway_5kt", "Is_Low_Ceiling_Risk", "Is_Severe_Convection_Risk",
    "Is_Thunderstorm_Risk", "Convective_Severity_Score", "Runway_Wet_Risk",
    "Forced_Runway_Swap_Risk", "Weather_Delay_Risk_Score", "Aviation_Operational_Risk_Score",
]

FEATURE_SETS = {
    "F0_base": F0_BASE,
    "F1_turnaround": F0_BASE + F1_TURNAROUND,
    "F2_lag": F0_BASE + F1_TURNAROUND + F2_LAG,
    "F3_weather": F0_BASE + F1_TURNAROUND + F2_LAG + WEATHER_FEATURES,
}

def available_features(columns, candidates):
    return [c for c in candidates if c in columns and c not in LEAKAGE_OR_LABEL_COLUMNS]

feature_audit_rows = []
for set_name, candidates in FEATURE_SETS.items():
    used = available_features(df.columns, candidates)
    missing = sorted(set(candidates) - set(used))
    leaked = sorted(set(used) & LEAKAGE_OR_LABEL_COLUMNS)
    feature_audit_rows.append({
        "feature_set": set_name,
        "candidate_count": len(candidates),
        "used_count": len(used),
        "missing_count": len(missing),
        "missing_features": ", ".join(missing),
        "leakage_overlap": ", ".join(leaked),
    })

feature_audit = pd.DataFrame(feature_audit_rows)
feature_audit.to_csv(REPORT_DIR / "02_feature_set_audit.csv", index=False)
display(feature_audit)


## 3. Time-Based Split

The benchmark uses a chronological split. By default, flights before March 2026 are train data and March 2026 is test data.


In [ ]:
train_df = df[df["Scheduled_Time"] < TEST_START].copy()
test_df = df[df["Scheduled_Time"] >= TEST_START].copy()

if train_df.empty or test_df.empty:
    raise ValueError("Time split produced an empty train or test set. Check TEST_START and Scheduled_Time coverage.")

split_profile = pd.DataFrame([
    {"split": "train", "rows": len(train_df), "start": train_df["Scheduled_Time"].min(), "end": train_df["Scheduled_Time"].max(), "target_median": train_df[TARGET].median(), "target_p95": train_df[TARGET].quantile(0.95)},
    {"split": "test", "rows": len(test_df), "start": test_df["Scheduled_Time"].min(), "end": test_df["Scheduled_Time"].max(), "target_median": test_df[TARGET].median(), "target_p95": test_df[TARGET].quantile(0.95)},
])
split_profile.to_csv(REPORT_DIR / "03_time_split_profile.csv", index=False)
display(split_profile)


## 4. Model Pipelines

All models use the same leakage-safe feature matrix. Numeric columns are median-imputed; categorical columns are imputed with `missing` and one-hot encoded. `Ridge` additionally scales numeric features.


In [ ]:
def make_preprocessor(X: pd.DataFrame, scale_numeric: bool = False) -> ColumnTransformer:
    categorical_cols = X.select_dtypes(include=["object", "string", "category"]).columns.tolist()
    numeric_cols = [c for c in X.columns if c not in categorical_cols]

    numeric_steps = [("imputer", SimpleImputer(strategy="median"))]
    if scale_numeric:
        numeric_steps.append(("scaler", StandardScaler()))

    numeric_pipe = Pipeline(numeric_steps)
    categorical_pipe = Pipeline([
        ("imputer", SimpleImputer(strategy="constant", fill_value="missing")),
        ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
    ])

    return ColumnTransformer(
        transformers=[
            ("num", numeric_pipe, numeric_cols),
            ("cat", categorical_pipe, categorical_cols),
        ],
        remainder="drop",
        sparse_threshold=0.0,
    )


def make_models(X_sample: pd.DataFrame):
    return {
        "DummyMedian": Pipeline([
            ("preprocess", make_preprocessor(X_sample, scale_numeric=False)),
            ("model", DummyRegressor(strategy="median")),
        ]),
        "Ridge": Pipeline([
            ("preprocess", make_preprocessor(X_sample, scale_numeric=True)),
            ("model", Ridge(alpha=10.0, random_state=RANDOM_STATE)),
        ]),
        "ExtraTrees": Pipeline([
            ("preprocess", make_preprocessor(X_sample, scale_numeric=False)),
            ("model", ExtraTreesRegressor(
                n_estimators=120,
                min_samples_leaf=10,
                max_features="sqrt",
                random_state=RANDOM_STATE,
                n_jobs=1,
            )),
        ]),
        "HistGradientBoosting": Pipeline([
            ("preprocess", make_preprocessor(X_sample, scale_numeric=False)),
            ("model", HistGradientBoostingRegressor(
                max_iter=160,
                learning_rate=0.05,
                l2_regularization=0.05,
                random_state=RANDOM_STATE,
            )),
        ]),
    }


def evaluate_predictions(y_true, y_pred):
    return {
        "MAE": mean_absolute_error(y_true, y_pred),
        "MedianAE": median_absolute_error(y_true, y_pred),
        "RMSE": float(np.sqrt(mean_squared_error(y_true, y_pred))),
        "R2": r2_score(y_true, y_pred),
    }


## 5. Run Ablation Benchmark

This trains each model on each feature set and evaluates on the March 2026 test split. `MAE` is the primary metric because it is directly interpretable in minutes.


In [ ]:
results = []
fitted_models = {}

for feature_set_name, candidates in FEATURE_SETS.items():
    features = available_features(df.columns, candidates)
    X_train = train_df[features].copy()
    y_train = train_df[TARGET].astype(float)
    X_test = test_df[features].copy()
    y_test = test_df[TARGET].astype(float)

    models = make_models(X_train)
    for model_name, pipeline in models.items():
        print(f"Training {feature_set_name} / {model_name} with {len(features)} features...")
        pipeline.fit(X_train, y_train)
        pred = np.clip(pipeline.predict(X_test), 0, 240)
        metrics = evaluate_predictions(y_test, pred)
        row = {
            "feature_set": feature_set_name,
            "model": model_name,
            "n_features": len(features),
            "train_rows": len(X_train),
            "test_rows": len(X_test),
            **metrics,
        }
        results.append(row)
        fitted_models[(feature_set_name, model_name)] = pipeline

results_df = pd.DataFrame(results).sort_values(["MAE", "RMSE"]).reset_index(drop=True)
results_df.to_csv(REPORT_DIR / "04_regression_benchmark_results.csv", index=False)
display(results_df)


## 6. Segment Diagnostics

Evaluate the best test model by airport and month. This helps show whether the model is learning a global pattern only, or whether performance differs across SGN/HAN/DAD and time.


In [ ]:
best = results_df.iloc[0]
best_key = (best["feature_set"], best["model"])
best_features = available_features(df.columns, FEATURE_SETS[best["feature_set"]])
best_model = fitted_models[best_key]

test_pred = np.clip(best_model.predict(test_df[best_features]), 0, 240)
predictions = test_df[["Airport", "Scheduled_Time", "IATA", "Aircraft_Type", "Category", TARGET]].copy()
predictions["prediction"] = test_pred
predictions["abs_error"] = (predictions[TARGET] - predictions["prediction"]).abs()
predictions["scheduled_month"] = predictions["Scheduled_Time"].dt.to_period("M").astype(str)

segment_rows = []
for group_cols in [["Airport"], ["scheduled_month"], ["Airport", "scheduled_month"]]:
    for keys, g in predictions.groupby(group_cols, dropna=False):
        if not isinstance(keys, tuple):
            keys = (keys,)
        row = {col: key for col, key in zip(group_cols, keys)}
        row.update({
            "grouping": "+".join(group_cols),
            "rows": len(g),
            "MAE": mean_absolute_error(g[TARGET], g["prediction"]),
            "MedianAE": median_absolute_error(g[TARGET], g["prediction"]),
            "RMSE": float(np.sqrt(mean_squared_error(g[TARGET], g["prediction"]))),
            "target_median": g[TARGET].median(),
            "prediction_median": g["prediction"].median(),
        })
        segment_rows.append(row)

segment_metrics = pd.DataFrame(segment_rows)
segment_metrics.to_csv(REPORT_DIR / "05_best_model_segment_metrics.csv", index=False)
predictions.to_csv(REPORT_DIR / "06_best_model_test_predictions.csv", index=False)

print("Best model:", best_key)
display(segment_metrics.sort_values(["grouping", "MAE"]))
display(predictions.sort_values("abs_error", ascending=False).head(20))


## 7. Interpretation Checklist

Use the generated CSV files in `Data crawl/Gold_layer/Audit/model_training/` when writing the benchmark section:

- `01_training_dataset_profile.csv`: confirms row counts and target distribution.
- `02_feature_set_audit.csv`: confirms which planned features exist in Gold.
- `03_time_split_profile.csv`: documents the chronological split.
- `04_regression_benchmark_results.csv`: main ablation/model comparison.
- `05_best_model_segment_metrics.csv`: airport/month diagnostics.
- `06_best_model_test_predictions.csv`: row-level predictions for error inspection.

The strongest valid result is not necessarily the lowest global MAE alone. Prefer a model that improves over `DummyMedian`, behaves consistently across airports, and does not rely on leakage-risk or raw identity columns.
